# Mamba (Selective State Space Model) — Time Series Forecasting

**Architecture**: Mamba SSM (Gu & Dao, 2023) — Selective State Space Model with input-dependent Δ, B, C

**Datasets**: Dow Jones, Lake Erie, Milk Production, S&P 500

**Runs**: 2 per dataset | **Epochs**: 100 | Same preprocessing as FLSTM notebooks

In [ ]:
# ============================================================
# PROCESS IDENTIFICATION
# ============================================================
import os
print(f"Process ID (PID): {os.getpid()}")

In [ ]:
# ============================================================
# NOTEBOOK TIMER — START
# ============================================================
import time as _timer_module
_NOTEBOOK_START_TIME = _timer_module.time()
print(f"Notebook execution started at: {_timer_module.strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ============================================================
# CPU ONLY Settings (Forced)
# ============================================================
import tensorflow as tf
import platform

try:
    tf.config.set_visible_devices([], 'GPU')
    print('Forcing CPU execution (disabled GPU visibility).')
except RuntimeError as e:
    print(e)

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt
import time

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
NUM_RUNS = 30
NUM_EPOCHS = 100
TEST_SPLIT = 60
LAG = 1

# Mamba hyperparameters
D_MODEL = 8       # Model dimension (matches FLSTM units=8)
D_STATE = 16      # SSM state dimension N
EXPAND = 2        # Expansion factor E

DATASETS = {
    'dow_jones': {
        'csv_path': '../content/monthly-closings-of-the-dowjones.csv',
        'display_name': 'Dow Jones',
    },
    'lake_erie': {
        'csv_path': '../content/monthly-lake-erie-levels-1921-19.csv',
        'display_name': 'Lake Erie',
    },
    'milk_production': {
        'csv_path': '../content/monthly-milk-production-pounds-p.csv',
        'display_name': 'Milk Production',
    },
    'sp500': {
        'csv_path': '../content/sp500.csv',
        'display_name': 'S&P 500',
    },
}

print(f'Config: runs={NUM_RUNS}, epochs={NUM_EPOCHS}, d_model={D_MODEL}, d_state={D_STATE}, expand={EXPAND}')

## Mamba Selective SSM Cell

Implements the full Mamba block as a single RNN cell:
- Input projection (expand by factor E)
- Conv1D surrogate + SiLU activation
- **Selective SSM** (S6): input-dependent Δ, B, C parameters
- Gated output (multiply with SiLU gate branch)
- Output projection (contract back to d_model)

In [ ]:
# ============================================================
# MAMBA SELECTIVE SSM CELL
# ============================================================
@tf.keras.utils.register_keras_serializable()
class MambaSSMCell(layers.Layer):
    """
    Combined Mamba block as a single RNN cell.
    
    Implements:
      - Input projection (expand)
      - Conv1D surrogate (Dense)
      - SiLU activation
      - Selective SSM (S6) with input-dependent Delta, B, C
      - Gated output (multiply with SiLU-activated gate branch)
      - Output projection (contract)
    
    State: SSM hidden state h, flattened to (d_inner * d_state,)
    """
    def __init__(self, d_model, d_state=16, expand=2, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.d_state = d_state
        self.expand = expand
        self.d_inner = d_model * expand
        
        # RNN cell interface
        self.state_size = self.d_inner * self.d_state
        self.output_size = d_model

    def build(self, input_shape):
        D = self.d_inner
        N = self.d_state
        
        # --- Input projection: project to 2*d_inner (main + gate) ---
        self.W_in = self.add_weight(shape=(input_shape[-1], D * 2),
            initializer='glorot_uniform', name='W_in')
        self.b_in = self.add_weight(shape=(D * 2,), initializer='zeros', name='b_in')
        
        # --- Conv1D surrogate (Dense) ---
        self.W_conv = self.add_weight(shape=(D, D),
            initializer='glorot_uniform', name='W_conv')
        self.b_conv = self.add_weight(shape=(D,), initializer='zeros', name='b_conv')
        
        # --- SSM A parameter: diagonal, log-space, S4D-Real init ---
        init_A_1d = np.log(np.arange(1, N + 1, dtype=np.float32))
        init_A_2d = np.tile(-init_A_1d, (D, 1))
        self.log_A = self.add_weight(shape=(D, N),
            initializer=tf.constant_initializer(init_A_2d), name='log_A')
        
        # --- s_B(x): input -> B ---
        self.W_B = self.add_weight(shape=(D, N),
            initializer='glorot_uniform', name='W_B')
        
        # --- s_C(x): input -> C ---
        self.W_C = self.add_weight(shape=(D, N),
            initializer='glorot_uniform', name='W_C')
        
        # --- s_Delta(x): input -> Delta ---
        self.W_delta = self.add_weight(shape=(D, D),
            initializer='glorot_uniform', name='W_delta')
        delta_bias_vals = np.log(np.exp(
            np.random.RandomState(42).uniform(0.001, 0.1, size=(D,)).astype(np.float32)
        ) - 1.0)
        self.b_delta = self.add_weight(shape=(D,),
            initializer=tf.constant_initializer(delta_bias_vals), name='b_delta')
        
        # --- Output projection: d_inner -> d_model ---
        self.W_out = self.add_weight(shape=(D, self.d_model),
            initializer='glorot_uniform', name='W_out')
        self.b_out = self.add_weight(shape=(self.d_model,),
            initializer='zeros', name='b_out')

    def call(self, inputs, states):
        D = self.d_inner
        N = self.d_state
        h = tf.reshape(states[0], (-1, D, N))
        
        # Input projection -> split main + gate
        xz = tf.matmul(inputs, self.W_in) + self.b_in
        x_main = xz[:, :D]
        z = xz[:, D:]
        
        # Main branch: conv -> SiLU
        x_main = tf.matmul(x_main, self.W_conv) + self.b_conv
        x_main = tf.nn.silu(x_main)
        
        # === Selective SSM (Algorithm 2) ===
        delta = tf.nn.softplus(tf.matmul(x_main, self.W_delta) + self.b_delta)
        B = tf.matmul(x_main, self.W_B)
        C = tf.matmul(x_main, self.W_C)
        
        # Discretize
        A = -tf.exp(self.log_A)
        delta_exp = tf.expand_dims(delta, -1)
        A_bar = tf.exp(delta_exp * A)
        B_bar = delta_exp * tf.expand_dims(B, 1)
        
        # Recurrence: h_t = A_bar * h_{t-1} + B_bar * x_t
        x_exp = tf.expand_dims(x_main, -1)
        h_new = A_bar * h + B_bar * x_exp
        
        # Output: y = sum(C * h, axis=-1)
        y_ssm = tf.reduce_sum(h_new * tf.expand_dims(C, 1), axis=-1)
        
        # Gate branch: SiLU
        z_gate = tf.nn.silu(z)
        
        # Multiply branches + project out
        y = y_ssm * z_gate
        output = tf.matmul(y, self.W_out) + self.b_out
        
        h_flat = tf.reshape(h_new, (-1, D * N))
        return output, [h_flat]

    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model, 'd_state': self.d_state, 'expand': self.expand})
        return config

print('MambaSSMCell defined.')

In [ ]:
# ============================================================
# MODEL BUILDER
# ============================================================
def build_mamba_model(input_dim, d_model=D_MODEL, d_state=D_STATE, expand=EXPAND, batch_size=1):
    cell = MambaSSMCell(d_model, d_state=d_state, expand=expand)
    rnn = layers.RNN(cell, return_sequences=False, stateful=True)
    
    inputs = tf.keras.Input(batch_shape=(batch_size, 1, input_dim))
    x = rnn(inputs)
    outputs = layers.Dense(1)(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
        loss='mean_squared_error'
    )
    return model

print('build_mamba_model defined.')

In [ ]:
# ============================================================
# PREPROCESSING (same as FLSTM notebooks)
# ============================================================
def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

def timeseries_to_supervised(data, lag):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag + 1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

print('Preprocessing functions defined.')

## Training & Evaluation — All 4 Datasets

In [ ]:
# ============================================================
# RUN ALL EXPERIMENTS
# ============================================================
all_results = []

for ds_key, ds_config in DATASETS.items():
    print(f"\n{'='*80}")
    print(f"  MAMBA SSM — {ds_config['display_name']}")
    print(f"  Runs: {NUM_RUNS}, Epochs: {NUM_EPOCHS}, d_model: {D_MODEL}")
    print(f"{'='*80}\n")
    
    # Load data
    series = pd.read_csv(ds_config['csv_path'], header=0, parse_dates=[0], index_col=0)
    raw_values = series.values.flatten()
    
    # Differencing
    diff_values = difference(raw_values, 1)
    
    # Supervised learning transformation
    supervised = timeseries_to_supervised(diff_values, LAG)
    train, test = supervised[:-TEST_SPLIT], supervised[-TEST_SPLIT:]
    
    # Scaling
    scaler = MinMaxScaler(feature_range=(-1, 1))
    train_scaled = scaler.fit_transform(train)
    test_scaled = scaler.transform(test)
    
    X_train_raw, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
    X_test_raw, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
    
    # Reshape for RNN: (samples, 1, features)
    X_train = X_train_raw.reshape((X_train_raw.shape[0], 1, X_train_raw.shape[1]))
    X_test = X_test_raw.reshape((X_test_raw.shape[0], 1, X_test_raw.shape[1]))
    input_dim = X_train.shape[2]
    
    print(f'  Train: {X_train.shape}, Test: {X_test.shape}')
    
    all_rmse, all_mse, all_nmse = [], [], []
    all_predictions = []
    
    for run in range(NUM_RUNS):
        print(f'\n--- RUN {run+1}/{NUM_RUNS} ---')
        run_start = time.time()
        
        np.random.seed(run)
        tf.random.set_seed(run)
        tf.keras.backend.clear_session()
        
        model = build_mamba_model(input_dim=input_dim, batch_size=1)
        
        if run == 0:
            model.summary()
        
        rnn_layer = model.layers[1]
        
        for epoch in range(NUM_EPOCHS):
            history = model.fit(X_train, y_train, epochs=1, batch_size=1, verbose=0, shuffle=False)
            rnn_layer.reset_states()
            if (epoch + 1) % 20 == 0:
                print(f'  Epoch {epoch+1}/{NUM_EPOCHS} — Loss: {history.history["loss"][0]:.6f}')
        
        # Warmup
        for i in range(len(X_train)):
            model.predict(X_train[i:i+1], batch_size=1, verbose=0)
        
        # Predict
        predictions = []
        for i in range(len(X_test)):
            yhat = model.predict(X_test[i:i+1], batch_size=1, verbose=0)
            row = list(X_test_raw[i]) + [yhat[0, 0]]
            inv = scaler.inverse_transform([row])[0, -1] + raw_values[len(train) + i]
            predictions.append(inv)
        
        actual = raw_values[-TEST_SPLIT:]
        rmse = sqrt(mean_squared_error(actual, predictions))
        mse = mean_squared_error(actual, predictions)
        meanV = np.mean(actual)
        dominator = np.linalg.norm(np.array(predictions) - meanV, 2)
        nmse = mse / np.power(dominator, 2)
        
        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)
        
        run_time = time.time() - run_start
        print(f'  Run {run+1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}, Time: {run_time:.1f}s')
    
    # Store results
    all_results.append({
        'dataset': ds_config['display_name'],
        'mean_rmse': np.mean(all_rmse), 'std_rmse': np.std(all_rmse),
        'mean_mse': np.mean(all_mse), 'std_mse': np.std(all_mse),
        'mean_nmse': np.mean(all_nmse), 'std_nmse': np.std(all_nmse),
        'best_rmse': min(all_rmse), 'best_mse': all_mse[np.argmin(all_rmse)],
        'best_nmse': all_nmse[np.argmin(all_rmse)],
        'all_rmse': all_rmse, 'all_mse': all_mse, 'all_nmse': all_nmse,
        'all_predictions': all_predictions, 'actual': actual,
    })
    
    print(f'\n  Mean RMSE: {np.mean(all_rmse):.6f} ± {np.std(all_rmse):.6f}')
    print(f'  Mean MSE:  {np.mean(all_mse):.6f} ± {np.std(all_mse):.6f}')
    print(f'  Mean NMSE: {np.mean(all_nmse):.10f} ± {np.std(all_nmse):.10f}')

print('\n\nAll experiments complete!')

## Consolidated Results Table

In [ ]:
# ============================================================
# FINAL CONSOLIDATED TABLE
# ============================================================
print('=' * 110)
print('  CONSOLIDATED RESULTS — MAMBA SSM (Selective State Space Model)')
print(f'  Runs: {NUM_RUNS} | Epochs: {NUM_EPOCHS} | d_model: {D_MODEL} | d_state: {D_STATE} | expand: {EXPAND}')
print('=' * 110)
print()

# Build a DataFrame for clean display
table_data = []
for r in all_results:
    table_data.append({
        'Dataset': r['dataset'],
        'Mean RMSE': f"{r['mean_rmse']:.6f}",
        'Std RMSE': f"{r['std_rmse']:.6f}",
        'Mean MSE': f"{r['mean_mse']:.6f}",
        'Std MSE': f"{r['std_mse']:.6f}",
        'Mean NMSE': f"{r['mean_nmse']:.10f}",
        'Std NMSE': f"{r['std_nmse']:.10f}",
        'Best RMSE': f"{r['best_rmse']:.6f}",
        'Best MSE': f"{r['best_mse']:.6f}",
        'Best NMSE': f"{r['best_nmse']:.10f}",
    })

results_df = pd.DataFrame(table_data)
results_df = results_df.set_index('Dataset')
display(results_df)

print()

# Per-run details
print('\nPer-Run Details:')
print('-' * 80)
run_data = []
for r in all_results:
    for i in range(NUM_RUNS):
        run_data.append({
            'Dataset': r['dataset'],
            'Run': i + 1,
            'RMSE': f"{r['all_rmse'][i]:.6f}",
            'MSE': f"{r['all_mse'][i]:.6f}",
            'NMSE': f"{r['all_nmse'][i]:.10f}",
        })

runs_df = pd.DataFrame(run_data)
runs_df = runs_df.set_index(['Dataset', 'Run'])
display(runs_df)

## Predictions vs Actual (Best Run per Dataset)

In [ ]:
# ============================================================
# PLOTS: Predictions vs Actual
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, r in enumerate(all_results):
    ax = axes[idx]
    best_idx = np.argmin(r['all_rmse'])
    actual = r['actual']
    best_pred = r['all_predictions'][best_idx]
    
    ax.plot(actual, label='Actual', color='blue', linewidth=1.5)
    ax.plot(best_pred, label=f'Predicted (Best Run {best_idx+1})', color='red',
            linewidth=1.5, linestyle='--')
    ax.set_title(f"Mamba SSM — {r['dataset']}\nRMSE: {r['all_rmse'][best_idx]:.4f}", fontsize=12)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Value')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Mamba SSM — Predictions vs Actual (Best of {NUM_RUNS} runs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# NOTEBOOK TIMER — END
# ============================================================
_NOTEBOOK_END_TIME = _timer_module.time()
_ELAPSED = _NOTEBOOK_END_TIME - _NOTEBOOK_START_TIME
print(f"Notebook execution finished at: {_timer_module.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total execution time: {_ELAPSED:.1f}s ({_ELAPSED/60:.1f} min)")